# Timed Auction — Bronze to Silver

This notebook processes raw timed auction bid data from the **bronze** layer (`company_db.bronze.timedauction`) and delivers cleaned, enriched records into the **silver** layer (`company_db.silver.timedauction`).

**What it does:**
1. Loads raw timed auction data from the bronze table.
2. Combines `first_name` and `last_name` into a `full_name` field; drops `currency`; renames `lot_#` to `lot_number`.
3. Deduplicates records on `item`, `lot_number`, `max_bid`, `user_email`, and `date_time`.
4. Splits the `auction_date` range string into separate `start_date` and `end_date` date columns.
5. Converts `date_time` to UTC (from the source system's local timezone) and extracts time parts: `month`, `month_day`, `hour`, `minute`, `seconds`, `week`.
6. Derives a `bid_date` column from the UTC `date_time`.
7. Creates the silver Delta table if it does not already exist.
8. Merges transformed records into silver using `date_time` + `user_email` + `lot_number` as the match key — updating existing rows and inserting new ones with audit timestamps.

**Source:** `company_db.bronze.timedauction`  
**Target:** `company_db.silver.timedauction` @ `abfss://auction@company.dfs.core.windows.net/silver/timedauction`

In [ ]:
catalog = dbutils.widgets.get("catalog")
container = dbutils.widgets.get("container")
storage_account = dbutils.widgets.get("storage_account")

In [ ]:
%run "../setup/002_Environment_variables"

In [ ]:
dbutils.widgets.text('id'," ")
batch_id = dbutils.widgets.get('id')
batch_id

In [ ]:
# Import PySpark SQL functions module — aliased as F for convenience throughout this notebook.
from src.transformations.timed_auction import transform_timed_auction

In [ ]:
# Read the raw auction summary dataset from the bronze layer.
# This is the source DataFrame that will be standardized and merged into the silver table.
timedauction = spark.table(f'{catalog}.bronze.timedauction').filter(F.col('batch_id')==batch_id)

In [ ]:
registeredusers = (
    spark.table(f"{catalog}.silver.registeredusers")
    .filter(F.col("batch_id") == batch_id)
    .select("email", "bidder_id", "user_id")
    .withColumnRenamed("email", "user_email")
)

In [ ]:
timedauction = timedauction.join(
    registeredusers,
    on="user_email",
    how="left"
)

In [ ]:
timedauction = transform_timed_auction(timedauction)

In [ ]:
# Join auction information
auction_sales_report = (
    spark.table(f"{catalog}.silver.auction_sales_report")
    .filter(F.col("batch_id") == batch_id)
    .select("start_date", "auction_id")
)


In [ ]:
timedauction = timedauction.join(
    auction_sales_report,
    on="start_date",
    how="left"
)

In [ ]:
# Join lot information
auctionresults = (
    spark.table(f"{catalog}.silver.auctionresults")
    .filter(F.col("batch_id") == batch_id)
    .select("lot_id", "lot_number")
)

In [ ]:
timedauction = timedauction.join(
    auctionresults,
    on="lot_number",
    how="left"
)

In [ ]:
# Register the transformed DataFrame as a temporary SQL view.
# This allows the downstream MERGE statement (SQL cell) to reference it as a source.
timedauction.createOrReplaceTempView('timedauction_view')

In [ ]:
spark.sql(
f"""
CREATE TABLE IF NOT EXISTS {catalog}.silver.timedauction 
(
auction_id int,
auction_name STRING,
item_name string,
lot_id int,
lot_number int,
current_bid double,
max_bid double,
user_id int,
bidder_id int,
first_name string,
last_name string,
full_name string, 
user_email string,
date_time timestamp,
path string, 
ingestion_timestamp timestamp,
start_date date, 
end_date date, 
month int, 
month_day string, 
hour int, 
minute int, 
seconds int, 
week string, 
bid_date date,
created_at TIMESTAMP,
updated_at TIMESTAMP,
batch_id string
)
USING DELTA
LOCATION 'abfss://{container}@{storage_account}.dfs.core.windows.net/silver/timedauction';""")

In [ ]:
spark.sql(
f"""
MERGE INTO {catalog}.silver.timedauction as target
USING timedauction_view as source
ON target.date_time = source.date_time
AND
target.user_email = source.user_email
AND
target.lot_number = source.lot_number
WHEN MATCHED 
AND target.batch_id >= source.batch_id 
THEN
   UPDATE SET 
   target.auction_id = source.auction_id,
   target.auction_name = source.auction_name,
   target.item_name = source.item,
   target.lot_id = source.lot_id,
   target.lot_number = source.lot_number,
   target.current_bid = source.current_bid,
   target.max_bid = source.max_bid,
   target.user_id = source.user_id,
   target.bidder_id = source.bidder_id,
   target.first_name = source.first_name,
   target.last_name = source.last_name,
   target.user_email = source.user_email,
   target.date_time = source.date_time,
   target.path = source.path,
   target.ingestion_timestamp = source.ingestion_timestamp,
   target.full_name = source.full_name,
   target.start_date = source.start_date,
   target.end_date = source.end_date,
   target.month = source.month,
   target.month_day = source.month_day,
   target.hour = source.hour,
   target.minute = source.minute,
   target.seconds = source.seconds,
   target.week = source.week,
   target.bid_date = source.bid_date,
   target.updated_at = current_timestamp(),
   target.batch_id = source.batch_id
WHEN NOT MATCHED THEN
   INSERT (auction_id, auction_name, item_name, lot_id, lot_number, current_bid, max_bid,user_id,bidder_id, first_name, last_name, user_email, date_time, path, ingestion_timestamp, full_name, start_date, end_date, month, month_day, hour, minute, seconds, week, bid_date, created_at, updated_at,batch_id)
   VALUES (source.auction_id,source.auction_name, source.item, source.lot_id, source.lot_number, source.current_bid, source.max_bid,source.user_id,source.bidder_id, source.first_name, source.last_name, source.user_email, source.date_time, source.path, source.ingestion_timestamp, source.full_name, source.start_date, source.end_date, source.month, source.month_day, source.hour, source.minute, source.seconds, source.week, source.bid_date, current_timestamp(), current_timestamp(),source.batch_id);
 """)